# Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from llm.interface.qwen import Qwen

In [2]:
import sqlite3
import pandas as pd
from tqdm import tqdm
from llm.prompts import clear_schema_system_prompt
from utils import format_schema_with_samples, parse_code_string

In [3]:
model = Qwen("llm/weight/qwen25-7b")
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [7]:
question = "Among the schools with the average score in Math over 560 in the SAT test, how many schools are in the bay area?"
target_schema = "['ID', 'NAME', 'RATING', 'PHONENUMBER', 'NO_OF_REVIEWS', 'ADDRESS']"

# Plan Generation

## Preparation

In [ ]:
from utils import read_jsonl
DATA_SRC = '../../data_src'
tables = {
    'Table_0': 'pandas_dfs/codebase_community/comments.csv',
    'Table_1': 'pandas_dfs/codebase_community/posts.csv',
    'Table_2': 'pandas_dfs/california_schools/satscores.csv',
}
tables_2 = {
    'table_11': 'table_11.csv',
    'table_5': 'table_5.csv',
}
MAX_COLS_TO_PROCESS = 10

# Ingest column descriptions
table_descs = {}
public_desc = read_jsonl(f'{DATA_SRC}/public.jsonl')
for table in tables_2:
    table_desc = [i for i in public_desc if i['table'] == table]
    for desc in table_desc:
        table_descs[table] = {}
        for col_info in desc['summary'].split(' | '):
            col_splits = col_info.split(': ')
            col_name = col_splits[0]
            col_summary = col_splits[1]
            table_descs[table][col_name] = col_summary

In [ ]:
clear_schema_system_prompt_neo = """You are given a column of a table, along with a description of what it likely represents. Please rename the column based on the following rules:

- Make the new column name explicit and easy to understand.
- Separate the tokens in the new name using whitespaces (e.g., 'restaurant name' instead of 'RestaurantName' or 'restaurant_name')
- Keep the use of abbreviations to a minimum (e.g., use average instead of avg).
- Keep the name accurate by not simplifying details such as age to age group (e.g., "5 years old" to "young").
- You can use symbols such as ">=" to simplify the name.

Output the name directly without any extra formatting, explanations, or text."""

In [ ]:
updated_schemas: list[str] = []
for table in tables_2:
    df = pd.read_csv(f'{DATA_SRC}/{tables_2[table]}')
    cols = df.columns
    new_cols = []
    for i in tqdm(range(MAX_COLS_TO_PROCESS)):
        col = cols[i]
        col_desc: str = table_descs[table][col]
        output = col
        if not col_desc.lower().startswith('no description'):
            msg = [
                {'role': 'system', 'content': clear_schema_system_prompt_neo},
                {'role': 'user', 'content': f'Column name: {col}\n\nColumn Description: {col_desc}'}
            ]
            output = model.chat(msg)
        print(output)
        new_cols.append(output)
    print(new_cols)
    # df.columns = new_cols
    # df.to_sql(table, conn, index=False, if_exists="replace")
    # updated_schemas.append(new_cols)

In [ ]:
# updated_schemas: list[str] = []
# for table in tqdm(tables_2):
#     df = pd.read_csv(f'{DATA_SRC}/{tables_2[table]}')
#     first_step_msg = [
#         {'role': 'system', 'content': clear_schema_system_prompt},
#         {'role': 'user', 'content': f'Table: {format_schema_with_samples(df)}'}
#     ]
#     output = model.chat(first_step_msg)
#     updated_schema = parse_code_string(output)
#     df.columns = updated_schema
#     df.to_sql(table, conn, index=False, if_exists="replace")
#     updated_schemas.append(updated_schema)

In [ ]:
for i in updated_schemas:
    print(i)

## Step 1: Get Base Table

In [ ]:
# updated_schemas = [
#     ["ID", "Post ID", "Score", "Text", "Creation Date", "User ID", "User Display Name"],
#     [
#         "ID",
#         "Post Type ID",
#         "Accepted Answer ID",
#         "Creation Date",
#         "Score",
#         "View Count",
#         "Body",
#         "Owner User ID",
#         "Last Activity Date",
#         "Title",
#         "Tags",
#         "Answer Count",
#         "Comment Count",
#         "Favorite Count",
#         "Last Editor User ID",
#         "Last Edit Date",
#         "Community Owned Date",
#         "Parent ID",
#         "Closed Date",
#         "Owner Display Name",
#         "Last Editor Display Name",
#     ],
#     [
#         "CD Number",
#         "School Type",
#         "School Name",
#         "District Name",
#         "County Name",
#         "Enrollment 12",
#         "Number of Test Takers",
#         "Average Score Reading",
#         "Average Score Math",
#         "Average Score Writing",
#         "Number of GE 1500",
#     ],
# ]

In [10]:
tables = ['yelp.csv', 'zomato.csv']
DATA_SRC = '../../data_src'

In [15]:
from utils import format_schema
available_tables = ""
for table_idx, table in enumerate(tables):
    df = pd.read_csv(f'{DATA_SRC}/{tables[table_idx]}')
    available_tables += f"\nTable_{table_idx}: ```{format_schema(df)}```\n"

In [40]:
plan_generator_first_step_system_prompt = """You are a helpful data scientist.

You will be provided with:
- A question in natural language.
- A list of available tables, each represented with its schema and a sample row.
- A target schema, which defines the supposedly relevant table to answer the question.

Your goal is to determine which table(s) among the available tables consists of the superset or the exact set of the target schema. If a single table exists, return "operation": "select_table" and specify the table.
However, if you think table join(s) are necessary/useful (e.g., there are tables with same schemas easily joinable), then return "operation": "join" and specify the necessary joins.

The output format for selecting a single table:
{
    "operation": "select_table",
    "tables_involved": ["Table_0"],
    "description": "Select Table_0."
}

While for joining tables:
{
    "operation": "join",
    "tables_involved": ["Table_0", "Table 1"],
    "description": "Join Table_0 with Table_1 on Table_0.Department ID and Table_1.DeptID"
}

Output your result strictly as a Python dictionary, without any extra formatting, explanations, or text. The output must be directly parseable as a Python dictionary."""

In [41]:
from llm.prompts import plan_generator_first_step_system_prompt
first_step_msg = [
    {'role': 'system', 'content': plan_generator_first_step_system_prompt},
    {'role': 'user', 'content': f'Question: Which restaurant has ratings above 3.0?\nAvailable Tables: {available_tables}\nTarget Schema: {target_schema}'}
]

In [42]:
first_step_out = model.chat(first_step_msg)

In [45]:
yelp = pd.read_csv('../../data_src/yelp.csv')
zomato = pd.read_csv('../../data_src/zomato.csv')

In [50]:
m = set(list(yelp['ID'].unique()))
n = set(list(zomato['ID'].unique()))

In [53]:
len(m)

5882

In [52]:
len(m - n)

5882

In [44]:
print(first_step_out)

{
    "operation": "select_table",
    "tables_involved": ["Table_0"],
    "description": "Select Table_0."
}


In [ ]:
from utils import format_schema
available_tables = ""
for table_idx, table in enumerate(tables):
    df = pd.read_csv(f'{DATA_SRC}/{tables[table]}', names=updated_schemas[table_idx], skiprows=1)
    available_tables += f"\n{table}: ```{format_schema(df)}```\n"

In [ ]:
from llm.prompts import plan_generator_first_step_system_prompt
first_step_msg = [
    {'role': 'system', 'content': plan_generator_first_step_system_prompt},
    {'role': 'user', 'content': f'Question: {question}\nAvailable Tables: {available_tables}\nTarget Schema: {target_schema}'}
]

In [ ]:
first_step_out = model.chat(first_step_msg)

In [ ]:
first_step_out = parse_code_string(first_step_out)

In [ ]:
print(first_step_out)

## Step 1.5: Extract Base Table from Memory

In [ ]:
# first_step_out = {
#     "operation": "select_table",
#     "tables_involved": ["Table_2"],
#     "description": "Select Table_2."
# }

In [ ]:
extract_base_sql = model.chat(
    [
        {
            "role": "user",
            "content": f"Convert this description: {first_step_out} to SQL code (sqlite3). Please answer directly; ensure that your output can be parsed directly as SQL code.",
        }
    ]
)

In [ ]:
print(extract_base_sql)

In [ ]:
base_table = pd.read_sql('SELECT * FROM Table_2;', conn)

In [ ]:
extract_column_table = base_table.to_sql('base_table', conn, index=False, if_exists="replace")

## Step 2: Column Projection

In [ ]:
col_projection_system_prompt = """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema that we will transform the source table into in a step-by-step manner.
- A column from the target schema as the current target column.

Your goal is to determine whether to select a certain column from SRC or extract information from certain column(s) from SRC to form the target column.

The output format for selecting a certain column:
{
    "operation": "select_column",
    "columns_involved": ["Restaurant ID"],
    "description": "Select SRC.Restaurant ID."
}

While for extracting information from certain column(s):
{
    "operation": "extract_column",
    "columns_involved": ["City", "ZIP Code"],
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`."
}

Output your result strictly as a Python dictionary, without any extra formatting, explanations, or text. The output must be directly parseable as a Python dictionary."""

In [ ]:
from ast import literal_eval
target_schema_parsed = literal_eval(target_schema)
second_step_msg = [
    {'role': 'system', 'content': col_projection_system_prompt},
    {'role': 'user', 'content': f'Source table: ```{format_schema_with_samples(base_table)}```\nTarget Schema: {target_schema_parsed}\nTarget Column: `{target_schema_parsed[2]}`'}
]

In [ ]:
second_step_output = model.chat(second_step_msg)

In [ ]:
print(second_step_output)

## Step 2a: Execute

In [ ]:
operations = [
    {
        "operation": "select_column",
        "columns_involved": ["CD Number"],
        "description": "Select SRC.CD Number.",
    },
    {
        "operation": "select_column",
        "columns_involved": ["School Name"],
        "description": "Select SRC.School Name.",
    },
    {
        "operation": "select_column",
        "columns_involved": ["Average Score Math"],
        "description": "Select SRC.`Average Score Math`.",
    },
    {
        "operation": "extract_column",
        "columns_involved": ["District Name", "County Name"],
        "description": "Check if SRC.District Name or SRC.County Name contains 'Bay Area'.",
    },
]

### Extract_Column Operation

In [ ]:
import pandas as pd
import sqlite3
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
base_table = pd.read_csv('base_table.csv')
base_table.to_sql('base_table', conn, if_exists='replace', index=False)
extract_col_table = pd.read_sql('SELECT "District Name", "County Name" FROM base_table;', conn)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from llm.interface.qwen import Qwen

In [ ]:
model = Qwen("llm/weight/qwen25-7b")
model.load_model()
model.load_tokenizer()

In [ ]:
extract_col_system_prompt = """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A column to be added to this table whose values depend on the other columns in the table.

Your goal is to determine the values of the new column for all rows. Ensure you consider **all provided columns together** rather than relying on a single column. For example, a city name may exist in multiple locations, but when paired with its corresponding province or county, ambiguity is reduced.

Output your result strictly as a Python list representing the new column values for all rows, without any extra formatting, explanations, or text. The output must be directly parseable as a Python list."""

In [ ]:
unique_ex_col_tbl = extract_col_table.drop_duplicates().reset_index(drop=True)
unique_ex_col_tbl = unique_ex_col_tbl[unique_ex_col_tbl['County Name'] == 'Alameda']

In [ ]:
rows = []
inc = 1
for i in range(0, len(unique_ex_col_tbl), inc):
    rows.append((i, i+inc))
rows[-1] = (rows[-1][0], len(unique_ex_col_tbl))

In [ ]:
from utils import format_schema_extensive, parse_code_string
from collections import Counter
from tqdm import tqdm
new_col_values = []
for row in tqdm(rows):
    extract_col_msg = [
        {'role': 'system', 'content': extract_col_system_prompt},
        {'role': 'user', 'content': f'Table ({row[1]-row[0]} rows): ```{format_schema_extensive(unique_ex_col_tbl, row[0], row[1])}```\nOverall Schema: {list(base_table.columns)}\nNew Column: `Is in Bay Area` (data type: boolean)'}
    ]
    sample_values = []
    temps = [0.1, 0.3, 0.5, 0.7, 0.9]
    for i in range(5):
        extract_col_output = model.chat(extract_col_msg, True, temps[i], 42)
        sample_values.extend(parse_code_string(extract_col_output))
    print(sample_values)
    counter = Counter(sample_values)
    mode = counter.most_common(1)[0][0]
    new_col_values.append(mode)

In [ ]:
results_cache = dict()
columns = unique_ex_col_tbl.columns
for idx, row in unique_ex_col_tbl.iterrows():
    vals = []
    for col in columns:
        vals.append(row[col])
    key = "_SEP_".join(vals)
    results_cache[key] = new_col_values[idx]

In [ ]:
actual_values = []
for idx, row in extract_col_table.iterrows():
    vals = []
    for col in columns:
        vals.append(row[col])
    key = "_SEP_".join(vals)
    actual_values.append(results_cache[key])

In [ ]:
y = base_table[["CD Number", "School Name", "Average Score Math"]]
y['Is in Bay Area'] = actual_values
y.to_csv('processor_output.csv', index=False)

## FINAL STEP: Answer the Question

In [ ]:
proc = pd.read_csv('processor_output.csv')

In [ ]:
qa_sys_prompt = """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A question over the table.

Your goal is to produce a SQL code (SQLite) to answer the question using the table.

Output your result strictly as a SQL code (SQLite) without any extra formatting, explanations, or text. The output must be directly parseable as a SQL code."""

In [ ]:
question = "Among the schools with the average score in Math over 560 in the SAT test, how many schools are in the bay area?"

In [ ]:
msg = [
    {'role': 'system', 'content': qa_sys_prompt},
    {'role': 'user', 'content': f'Table `proc`: ```{format_schema_extensive(proc, 0, 3)}```\nQuestion: {question}'},
]

In [ ]:
model.chat(msg)

In [ ]:
proc.to_sql('proc', conn, index=False, if_exists="replace")

In [ ]:
result = pd.read_sql("""SELECT COUNT(*) FROM proc WHERE "Average Score Math" > 560 AND "Is in Bay Area" = True;""", conn)

In [ ]:
result